In [1]:
from pyspark.sql import SparkSession
spark = SparkSession. \
builder. \
master("local[4]"). \
appName("JOIN in spark"). \
getOrCreate()

In [2]:
order_schema = "order_id long, order_date string, customer_id long, order_status string"

In [3]:
df = spark.read \
.format("csv") \
.schema(order_schema) \
.load("D:\Learn-spark\learn-spark-maide\orders_large.csv")

In [4]:
df.show(20)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11427|     PROCESSING|
|       2|2013-07-25 00:00:...|       9381|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|       3606|         CLOSED|
|       4|2013-07-25 00:00:...|       4253|     PROCESSING|
|       5|2013-07-25 00:00:...|       7018|PENDING_PAYMENT|
|       6|2013-07-25 00:00:...|       3989|     PROCESSING|
|       7|2013-07-25 00:00:...|       3833|PENDING_PAYMENT|
|       8|2013-07-25 00:00:...|      10766|     PROCESSING|
|       9|2013-07-25 00:00:...|       8038|       COMPLETE|
|      10|2013-07-25 00:00:...|      10771|       COMPLETE|
|      11|2013-07-25 00:00:...|       6451|       COMPLETE|
|      12|2013-07-25 00:00:...|       6746|       COMPLETE|
|      13|2013-07-25 00:00:...|       5917|         CLOSED|
|      14|2013-07-25 00:00:...|       46

In [5]:
customer_schema = "customerid long, customer_fname string, customer_lname string, user_name string, password string, address string, city string, state string, pincode string"

In [6]:
customer_df = spark.read \
.format("csv") \
.schema(customer_schema) \
.load("D:\Learn-spark\learn-spark-maide\customers_join.csv")

In [8]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
#Set up mac dinh : 10485760b

'10485760b'

In [9]:
df.join(customer_df, df.customer_id == customer_df.customerid, "inner").write.format("noop").mode("overwrite").save()

In [10]:
df.join(customer_df, df.customer_id == customer_df.customerid, "inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [customer_id#2L], [customerid#30L], Inner, BuildRight, false
   :- Filter isnotnull(customer_id#2L)
   :  +- FileScan csv [order_id#0L,order_date#1,customer_id#2L,order_status#3] Batched: false, DataFilters: [isnotnull(customer_id#2L)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/Learn-spark/learn-spark-maide/orders_large.csv], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:bigint,order_date:string,customer_id:bigint,order_status:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=91]
      +- Filter isnotnull(customerid#30L)
         +- FileScan csv [customerid#30L,customer_fname#31,customer_lname#32,user_name#33,password#34,address#35,city#36,state#37,pincode#38] Batched: false, DataFilters: [isnotnull(customerid#30L)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/Learn-spark/learn-

Vì 2 phep join còn lại cần file customer lớn, mà mình cố tình chạy thì spark sẽ tối ưu về Broadcast join. Nên là sẽ tắt autoBroadcastJoinThreshold đi

In [13]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

In [14]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

'-1'

In [15]:
df.join(customer_df, df.customer_id == customer_df.customerid, "inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [customer_id#2L], [customerid#30L], Inner
   :- Sort [customer_id#2L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#2L, 200), ENSURE_REQUIREMENTS, [plan_id=115]
   :     +- Filter isnotnull(customer_id#2L)
   :        +- FileScan csv [order_id#0L,order_date#1,customer_id#2L,order_status#3] Batched: false, DataFilters: [isnotnull(customer_id#2L)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/D:/Learn-spark/learn-spark-maide/orders_large.csv], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:bigint,order_date:string,customer_id:bigint,order_status:string>
   +- Sort [customerid#30L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customerid#30L, 200), ENSURE_REQUIREMENTS, [plan_id=116]
         +- Filter isnotnull(customerid#30L)
            +- FileScan csv [customerid#30L,customer_fname#31,customer_lname#32,user_name#33,pas

In [16]:
df.join(customer_df, df.customer_id == customer_df.customerid, "inner").write.format("noop").mode("overwrite").save()

Nếu không .hint() thì thường sẽ là sort merge join, để chắc chắn thì
df.join(customer_df.hint("shuffle_merge"), df.customer_id == customer_df.customerid, "inner").write.format("noop").mode("overwrite").save()

Shuffle hash join

In [17]:
df.join(customer_df.hint("shuffle_hash"), df.customer_id == customer_df.customerid, "inner").write.format("noop").mode("overwrite").save()